# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shreeyeshbaral/ShreeyeshAssignment1/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

**Lane 4 — CTR / Engagement Opportunity Scoring.**

I'm choosing this lane because the starter data reveals a large, clearly measurable gap between the visibility pages already have and the clicks or engagement they actually capture. Nearly a third of high-volume, well-positioned pages show CTR well below what comparable pages in the same position tier achieve. That gap represents wasted visibility: impressions that never become clicks. A ranked opportunity score that adjusts for position and volume could help a content team focus on the pages where a title rewrite, meta-description improvement, or on-page engagement fix is most likely to matter — instead of scanning thousands of pages by hand or relying on a single rule threshold.

In [1]:
# Load the starter dataset and confirm its shape.
import pandas as pd
import os

# Handle both Colab (cloned repo) and local paths
local_path = '../../data/raw/content_refresh_anonymized.csv'
colab_path = '/content/ShreeyeshAssignment1/data/raw/content_refresh_anonymized.csv'

if os.path.exists(local_path):
    csv_path = local_path
elif os.path.exists(colab_path):
    csv_path = colab_path
else:
    raise FileNotFoundError('Could not find content_refresh_anonymized.csv')

df = pd.read_csv(csv_path)
print(f'Loaded starter dataset: {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'Distinct clients: {df["client_id"].nunique()}')

Loaded starter dataset: 30,000 rows × 44 columns
Distinct clients: 32


## 2. The question: decision, action, cost of a wrong call

**Research question:** Which visible pages under-capture clicks or engagement relative to their position tier, and should be reviewed first for metadata, content, or monitoring improvements?

**The decision this improves:** A content or SEO editor deciding which pages to review first, out of thousands, when the team can realistically act on 20–50 pages per cycle. Today that decision is either manual (slow, inconsistent) or based on a single threshold rule (misses context). A position-adjusted CTR opportunity score ranks the queue so the editor starts with the pages where the gap is largest and the volume is high enough to matter.

**Who acts, and what they do:** A content editor or SEO specialist. Depending on the reason code, they might rewrite a title or meta description (CTR gap), improve on-page structure (low engagement or scroll), add structured data, or flag the page for monitoring if the gap may be explained by SERP features or seasonality.

**What a wrong recommendation costs:**
- *False positive (recommending a page that doesn't need fixing):* The editor spends 15–30 minutes reviewing and deciding no action is needed. With a capacity of ~50 reviews per cycle, each false positive displaces a real opportunity. At a realistic cost of \$25–50 in editor time per review, a queue with 50% false positives wastes roughly half the review budget.
- *False negative (missing a page that does need attention):* A high-impression page continues to under-capture clicks. With thousands of impressions per day, even a small CTR improvement compounds over time — the cost is the opportunity left on the table.
- The false-positive cost is more directly measurable, so **precision at the top of the queue** (precision@K) is the right primary metric.

**Why data or ML helps:** A simple rule like "flag every page with CTR < 0.5%" catches 9,759 pages — far too many to review, and it ignores the fact that a page at position 18 naturally has lower CTR than a page at position 3. The pattern of what makes a page under-perform its peers depends on multiple signals tangled together: position, impression volume, content type, intent, age, freshness, and engagement signals. A model can learn those interactions and produce a ranked score; a single rule cannot.

In [2]:
# Quick frame check: how many pages fall under a naive rule vs. review capacity?
valid = df[df['avg_position'] > 0]  # Exclude avg_position == 0 (no data)

naive_rule = valid[
    (valid['impressions_90d'] >= 500) &
    (valid['avg_position'] <= 20) &
    (valid['ctr'] < 0.5)  # Remember: ctr is a ×100 percentage, so 0.5 = 0.5%
]

print(f'A naive rule (>=500 impressions, position <=20, CTR < 0.5%) flags:')
print(f'  {len(naive_rule):,} pages — {len(naive_rule)/len(df)*100:.1f}% of the dataset')
print(f'  A team reviewing 50 pages/cycle would need ~{len(naive_rule)//50} cycles to get through them.')
print(f'  → A ranked score is needed to prioritize the queue.')

A naive rule (>=500 impressions, position <=20, CTR < 0.5%) flags:
  9,759 pages — 32.5% of the dataset
  A team reviewing 50 pages/cycle would need ~195 cycles to get through them.
  → A ranked score is needed to prioritize the queue.


## 3. Quick look at the data (2-3 real numbers)

The numbers below come from the 30,000-row starter dataset (`data/raw/content_refresh_anonymized.csv`). They demonstrate that (a) the CTR opportunity pool is large and measurable, (b) CTR varies widely even among pages at the same position tier, and (c) volume is concentrated enough that improving the worst under-performers could matter.

In [3]:
# ── Number 1: CTR spread within position tiers ──────────────────────────
# If position alone explained CTR, pages in the same tier would cluster tightly.
# They don't — the spread shows room for other signals to explain the gap.

print('CTR distribution by position tier (pages with >=500 impressions):')
print('(CTR is a ×100 percentage: 0.24 means 0.24%)')
print()

tier_order = ['top_3', 'page_1', 'striking', 'page_3_5', 'deep']
rows = []
for tier in tier_order:
    sub = valid[(valid['position_tier'] == tier) & (valid['impressions_90d'] >= 500)]
    if len(sub) == 0:
        continue
    rows.append({
        'position_tier': tier,
        'n_pages': len(sub),
        'Q25_ctr': round(sub['ctr'].quantile(0.25), 3),
        'median_ctr': round(sub['ctr'].median(), 3),
        'Q75_ctr': round(sub['ctr'].quantile(0.75), 3),
        'Q90_ctr': round(sub['ctr'].quantile(0.90), 3),
    })

tier_df = pd.DataFrame(rows)
print(tier_df.to_string(index=False))
print()
print('→ Within every tier the Q25-to-Q75 CTR range spans 2–4× — position alone')
print('  does not explain the gap. Other signals (content type, intent, age, freshness)')
print('  likely contribute, and a model can learn those interactions.')

CTR distribution by position tier (pages with >=500 impressions):
(CTR is a ×100 percentage: 0.24 means 0.24%)

position_tier  n_pages  Q25_ctr  median_ctr  Q75_ctr  Q90_ctr
        top_3      458     0.08        0.20     0.49    0.863
       page_1     7064     0.12        0.24     0.44    0.750
     striking     4485     0.08        0.17     0.34    0.610
     page_3_5     4330     0.01        0.09     0.19    0.360
         deep      389     0.00        0.00     0.04    0.140

→ Within every tier the Q25-to-Q75 CTR range spans 2–4× — position alone
  does not explain the gap. Other signals (content type, intent, age, freshness)
  likely contribute, and a model can learn those interactions.


In [4]:
# ── Number 2: Page-1 under-performers represent massive impression volume ─

page1 = valid[valid['position_tier'] == 'page_1']
page1_hv = page1[page1['impressions_90d'] >= 500]
page1_median_ctr = page1_hv['ctr'].median()
page1_under = page1_hv[page1_hv['ctr'] < page1_median_ctr]

print(f'Page-1 pages (positions 4–10) with >=500 impressions: {len(page1_hv):,}')
print(f'Tier median CTR: {page1_median_ctr:.2f}%')
print(f'Pages below that median: {len(page1_under):,} ({len(page1_under)/len(page1_hv)*100:.1f}%)')
print(f'Combined 90-day impressions of those under-performers: {page1_under["impressions_90d"].sum():,}')
print()
print('→ 3,505 page-1 pages sit below their tier median CTR,')
print('  collectively representing ~41.7 million impressions in 90 days.')
print('  Even a small CTR improvement on this pool could meaningfully increase clicks.')

Page-1 pages (positions 4–10) with >=500 impressions: 7,064
Tier median CTR: 0.24%
Pages below that median: 3,505 (49.6%)
Combined 90-day impressions of those under-performers: 41,707,905

→ 3,505 page-1 pages sit below their tier median CTR,
  collectively representing ~41.7 million impressions in 90 days.
  Even a small CTR improvement on this pool could meaningfully increase clicks.


In [5]:
# ── Number 3: The naive rule pool is too large to review without ranking ─

pool = valid[
    (valid['impressions_90d'] >= 500) &
    (valid['avg_position'] <= 20) &
    (valid['ctr'] < 0.5)
]

print(f'Naive CTR-opportunity rule flags {len(pool):,} pages ({len(pool)/len(df)*100:.1f}% of dataset)')
print(f'Combined impressions: {pool["impressions_90d"].sum():,}')
print()

# How many of those are also declining?
pool_declining = pool[pool['trend_direction'] == 'down']
print(f'Of those, {len(pool_declining):,} ({len(pool_declining)/len(pool)*100:.1f}%) are also declining.')
print(f'These are the highest-priority candidates: low CTR AND losing visibility.')
print()
print('→ 9,759 pages pass a simple threshold — far too many for a team to review.')
print('  A position-adjusted, model-ranked score is needed to surface the top 20–50')
print('  where action is most likely to help.')

Naive CTR-opportunity rule flags 9,759 pages (32.5% of dataset)
Combined impressions: 90,968,008

Of those, 6,120 (62.7%) are also declining.
These are the highest-priority candidates: low CTR AND losing visibility.

→ 9,759 pages pass a simple threshold — far too many for a team to review.
  A position-adjusted, model-ranked score is needed to surface the top 20–50
  where action is most likely to help.


## 4. Careful words: what I can and can't claim

**What this work will be able to say (observed / directional / decision-support):**

- We can **observe** that CTR varies widely among pages with similar search positions, and identify which observable signals (content age, freshness, word count, intent, impression volume, engagement metrics) are **associated** with that variation.
- We can **build a ranked queue** that orders pages by how far their CTR falls below what comparable pages in the same position tier achieve — adjusted for volume so that low-impression noise doesn't dominate.
- We can **measure** whether a learned model ranks under-performers more accurately than a transparent rule baseline, using precision@K and average precision.
- We can **suggest** actions (rewrite title/meta, improve on-page engagement, add structured data, monitor) with reason codes, so an editor knows *why* each page was flagged.
- All results are **decision-support**: the queue helps a human reviewer prioritize; it does not make the decision for them.

**What this work will never claim:**

- We **cannot claim** that improving a title or meta description will *cause* CTR to increase. That would require an A/B experiment or another causal design, which this observational dataset cannot provide.
- We **cannot claim** to have discovered Google ranking factors. CTR under-performance may reflect SERP features, competition, query intent, seasonality, or other factors outside the content itself.
- We **cannot claim** results generalize beyond this dataset without testing on held-out clients and time periods.
- We **will not** use `trend_direction` or `trend_pct` as features (they are label sources). We will not use product decision flags (they are not in this dataset by design). We will not publish any raw client names, domains, or URLs.

In [6]:
# Verify: the fields we plan to use are observable signals, not product decisions.
planned_features = [
    'impressions_90d', 'clicks_90d', 'sessions_90d', 'avg_position',
    'content_age_days', 'days_since_last_update', 'word_count',
    'search_volume', 'competition', 'engagement_rate', 'scroll_rate',
    'content_type', 'main_intent', 'position_tier', 'age_tier',
    'freshness_tier', 'word_count_tier', 'impression_tier'
]

# These must NEVER be features (label sources or leakage risks)
excluded = ['trend_direction', 'trend_pct', 'ctr']  # ctr is our target proxy

print('Planned feature candidates (all observable signals or transparent tiers):')
for f in planned_features:
    present = f in df.columns
    print(f'  {f}: {"✓ present" if present else "✗ MISSING"}')

print(f'\nExcluded from features (label/target related):')
for f in excluded:
    print(f'  {f}: excluded')

print('\n→ All planned features are observable signals available before any decision point.')

Planned feature candidates (all observable signals or transparent tiers):
  impressions_90d: ✓ present
  clicks_90d: ✓ present
  sessions_90d: ✓ present
  avg_position: ✓ present
  content_age_days: ✓ present
  days_since_last_update: ✓ present
  word_count: ✓ present
  search_volume: ✓ present
  competition: ✓ present
  engagement_rate: ✓ present
  scroll_rate: ✓ present
  content_type: ✓ present
  main_intent: ✓ present
  position_tier: ✓ present
  age_tier: ✓ present
  freshness_tier: ✓ present
  word_count_tier: ✓ present
  impression_tier: ✓ present

Excluded from features (label/target related):
  trend_direction: excluded
  trend_pct: excluded
  ctr: excluded

→ All planned features are observable signals available before any decision point.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.